In [0]:
storage_account_name = "segrid"
storage_account_key = storage_account_key = "<STORAGE_ACCOUNT_KEY>"  

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

processed_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/energy_readings/"
curated_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/energy_readings/"

df = spark.read.parquet(processed_path)
print(f"Rows read from processed: {df.count()}")

Rows read from processed: 2075259


In [0]:
from pyspark.sql.functions import col, expr, trim
# missing value and type casting
numeric_cols = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

df_typed = df
for c in numeric_cols:
    df_typed = df_typed.withColumn(
        c, expr(f"try_cast(trim({c}) as double)")
    )

In [0]:
from pyspark.sql.functions import to_timestamp, concat_ws, hour, dayofweek, when as w

df_ts = df_typed.withColumn(
    "reading_timestamp",
    to_timestamp(concat_ws(" ", col("Date"), col("Time")), "d/M/yyyy H:mm:ss")
).select(
    col("reading_timestamp"),
    col("Global_active_power").alias("global_active_power_kw"),
    col("Global_reactive_power").alias("global_reactive_power_kw"),
    col("Voltage").alias("voltage_v"),
    col("Global_intensity").alias("global_intensity_a"),
    col("Sub_metering_1").alias("sub_metering_kitchen_wh"),
    col("Sub_metering_2").alias("sub_metering_laundry_wh"),
    col("Sub_metering_3").alias("sub_metering_waterheater_ac_wh"),
)

In [0]:
df_features = (
    df_ts
    .withColumn(
        "unmetered_power_wh",
        (col("global_active_power_kw") * 1000 / 60)
        - (col("sub_metering_kitchen_wh") + col("sub_metering_laundry_wh") + col("sub_metering_waterheater_ac_wh"))
    )
    .withColumn("hour_of_day", hour(col("reading_timestamp")))
    .withColumn("is_weekend", dayofweek(col("reading_timestamp")).isin([1, 7]))  # 1=Sun, 7=Sat
    .withColumn("day_type", w(dayofweek(col("reading_timestamp")).isin([1, 7]), "Weekend").otherwise("Weekday"))
)

In [0]:
from pyspark.sql.functions import lit, when as w
# voltage anomaly
df_anomaly = (
    df_features
    .withColumn(
        "anomaly_type",
        w((col("voltage_v") < 220) | (col("voltage_v") > 256), lit("Voltage_Anomaly"))
        .otherwise(lit(None))
    )
    .withColumn(
        "severity",
        w(col("anomaly_type").isNull(), lit(None))
        .when((col("voltage_v") < 220 * 0.85) | (col("voltage_v") > 256 * 1.2), lit("High"))
        .otherwise(lit("Medium"))
    )
)

In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_tagged = (
    df_anomaly
    .withColumn("ingestion_source", lit("Batch"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

window_spec = Window.partitionBy("reading_timestamp").orderBy("reading_timestamp")
df_deduped = (
    df_tagged
    .withColumn("row_num", row_number().over(window_spec))
)

df_curated = df_deduped.filter(col("row_num") == 1).drop("row_num")
df_duplicates_log = df_deduped.filter(col("row_num") > 1).drop("row_num")

print(f"Curated rows: {df_curated.count()}")
print(f"Duplicate rows logged: {df_duplicates_log.count()}")

df_curated.write.mode("overwrite").parquet(curated_path)
df_duplicates_log.write.mode("overwrite").parquet(f"{curated_path.rstrip('/')}_duplicates_log/")

display(df_curated.limit(10))

Curated rows: 2075259
Duplicate rows logged: 0


reading_timestamp,global_active_power_kw,global_reactive_power_kw,voltage_v,global_intensity_a,sub_metering_kitchen_wh,sub_metering_laundry_wh,sub_metering_waterheater_ac_wh,unmetered_power_wh,hour_of_day,is_weekend,day_type,anomaly_type,severity,ingestion_source,ingestion_timestamp
2006-12-16T17:26:00Z,5.374,0.498,233.29,23.0,0.0,2.0,17.0,70.56666666666666,17,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T17:27:00Z,5.388,0.502,233.74,23.0,0.0,1.0,17.0,71.8,17,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T17:31:00Z,3.7,0.52,235.22,15.8,0.0,1.0,17.0,43.666666666666664,17,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T17:40:00Z,3.27,0.152,236.73,13.8,0.0,0.0,17.0,37.5,17,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T17:45:00Z,7.706,0.0,230.98,33.2,0.0,0.0,17.0,111.43333333333334,17,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T17:48:00Z,4.474,0.0,234.96,19.4,0.0,0.0,17.0,57.56666666666666,17,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T17:52:00Z,3.258,0.0,235.49,13.8,0.0,0.0,17.0,37.3,17,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T18:02:00Z,2.772,0.118,238.28,11.6,0.0,0.0,17.0,29.200000000000003,18,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T18:04:00Z,4.928,0.202,235.01,21.0,0.0,37.0,16.0,29.13333333333334,18,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
2006-12-16T18:07:00Z,6.474,0.144,231.85,27.8,0.0,37.0,16.0,54.900000000000006,18,true,Weekend,null,null,Batch,2026-07-16T07:09:24.437236Z
